# 9주차 3차시: Self-RAG (자가 평가 기반 검색-생성 루프)

| 주제 | 내용 |
|---|---|
| Self-RAG 개념 | 검색/생성/평가를 LLM이 자체 판단 |
| SelfRAGState 설계 | question, documents, filtered_documents, generation 등 |
| retrieve & generate | 검색 노드, RAG 프롬프트 기반 생성 |
| grade_relevance | 문서별 관련성 yes/no 평가, _Relevance 모델 |
| 조건부 재검색 | relevance_router, 관련 문서 없으면 retrieve 재시도 |
| query rewrite | 검색 실패 시 동의어/구체화로 쿼리 재작성 |
| 환각 평가 | grade_hallucination, _Hallucination 모델 |

In [ ]:
!pip install langgraph

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.1 which is incompatible.


In [ ]:
import os
from typing import TypedDict, Annotated
# from dotenv import load_dotenv
# load_dotenv()

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage, SystemMessage, AIMessage, ToolMessage, BaseMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
import pandas as pd
llm = ChatOpenAI(model="gpt-4o-mini")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
import os
from typing import TypedDict, List, Literal, Optional
# from dotenv import load_dotenv
# load_dotenv()
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.callbacks import BaseCallbackHandler
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Self-RAG 개념
- 참고: https://arxiv.org/pdf/2310.11511
- 검색해온 후, 관련 있는지, 일부분 관련 있는지, 지지하는지 등 평가
- 자기 검열을 하는 RAG
- 알고리즘 참고
  - 답변이 문서와 관련 없는 내용을 출력했다면 할루시네이션으로 간주
  - IsUse: 답변 퀄리티가 도움이 많이 되었는지
  - 등등..

- adaptive RAG : 쿼리 -> 검색 수행 여부 판단
- self RAG : retrieved 문서에 대해 답변 유용성 등 판단하여 루프를 돌림

## SelfRAGState & 문서 준비

In [ ]:
class SelfRAGState(TypedDict):
  question : str # 사용자 질문, state가 업데이트되더라도 question은 잘 안바뀜
  rewritten_question: str # 같은 쿼리를 같은 모델에다 던지면 비슷한 답변... 답변이 잘 안나온다면 query를 새로 작성해서 던짐(self-rag의 원래 논문에는 없음 (C-RAG, corrective rag))
  documents : List[Document] # 반환된 문서
  filtered_documents : List[Document] # 관련도 있는 문서 (필터됨)
  generation : str # 생성된 LLM 답변
  relevance_grades : List[str] # 문서별 관련 여부
  hallucination_score : str # 환각 여부
  usefulness_score : int # 1-5점, 유용성
  relevance_retries : int # 문서 검색 했으나 관련도 검사하여 필터링했으나, 개수가 부족하면 다시 검색
  generation_retries : int
  max_tries : int # 무한루프 방지



In [ ]:
builder = StateGraph(SelfRAGState)
builder.add_edge(START,END)
app = builder.compile()

In [ ]:
DOCS = [
    Document(page_content="Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.",
             metadata={"src": "selfrag-intro"}),
    Document(page_content="원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.",
             metadata={"src": "selfrag-paper"}),
    Document(page_content="구현체는 LLM-as-judge 패턴(with_structured_output)으로 reflection token을 대신합니다.",
             metadata={"src": "selfrag-impl"}),
    Document(page_content="관련성 평가에서 yes 문서만 컨텍스트로 추리고, 모두 no면 쿼리 재작성 또는 재검색.",
             metadata={"src": "selfrag-relevance"}),
    Document(page_content="환각 평가는 답변이 문서에 근거하는지 확인합니다. yes(환각)면 다시 생성합니다.",
             metadata={"src": "selfrag-hallucination"}),
    Document(page_content="유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.",
             metadata={"src": "selfrag-usefulness"}),
    Document(page_content="LangGraph는 StateGraph로 노드와 조건부 엣지를 조립해 평가 루프를 자연스럽게 표현합니다.",
             metadata={"src": "langgraph"}),
    Document(page_content="쿼리 재작성은 검색 실패 시 동의어/구체화로 쿼리를 바꿔 다시 검색하는 폴백 패턴입니다.",
             metadata={"src": "query-rewrite"}),
]
vectorstore = FAISS.from_documents(DOCS, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### retrieve 노드

In [ ]:
def retrieve_node(state: SelfRAGState) -> dict:
  docs = retriever.invoke(state['question'])
  return {'documents': docs}

In [ ]:
INIT = {"question": "", "rewritten_question": "",
        "documents": [], "filtered_documents": [], "generation": "",
        "relevance_grades": [], "hallucination_score": "",
        "usefulness_score": 0,
        "relevance_retries": 0, "generation_retries": 0, "max_retries": 4}

In [ ]:
builder = StateGraph(SelfRAGState)

builder.add_node('retrieve', retrieve_node)
builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', END)
app = builder.compile()

In [ ]:
result = app.invoke({**INIT, "question" : 'self-rag의 환각 평가는 어떻게 하나요?'}) # **는 딕셔너리를 풀어주는 기능

In [ ]:
result

{'question': 'self-rag의 환각 평가는 어떻게 하나요?',
 'rewritten_question': '',
 'documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='2a6f31d6-9e76-447d-a6a3-6a07242cc066', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='afc716b5-297f-43e6-b611-7777c39a00eb', metadata={'src': 'selfrag-hallucination'}, page_content='환각 평가는 답변이 문서에 근거하는지 확인합니다. yes(환각)면 다시 생성합니다.')],
 'filtered_documents': [],
 'generation': '',
 'relevance_grades': [],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

### generate 노드 (RAG_PROMPT)

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '다음 문서를 근거로 한국어로 간단히 답하세요. 문서에 없으면 "모름"이라고 답하세요'),
    ('human', '질문 : {question}\n\n문서:\n{context}')
])

def generate_node(state: SelfRAGState) -> dict:
  ctx = '\n'.join(f" - {d.page_content}" for d in state['documents']) # state에 있는 document들에 대해서 d.pagecontent를 개행문자 기준 합쳐라
  msg = RAG_PROMPT.format_messages(question = state['question'], context = ctx)

  return {'generation' : llm.invoke(msg).content}

In [ ]:
builder = StateGraph(SelfRAGState)

builder.add_node('retrieve', retrieve_node)
builder.add_node('generate', generate_node)
builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'generate')
builder.add_edge('generate', END)
app = builder.compile()

In [ ]:
result = app.invoke({**INIT, 'question' : 'self-rag의 유용성 평가 기준은?'})

In [ ]:
result

{'question': 'self-rag의 유용성 평가 기준은?',
 'rewritten_question': '',
 'documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='2a6f31d6-9e76-447d-a6a3-6a07242cc066', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='32bfe861-8f3e-46f3-8a25-5a5aa6676cc7', metadata={'src': 'selfrag-usefulness'}, page_content='유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.')],
 'filtered_documents': [],
 'generation': 'self-RAG의 유용성 평가 기준은 답변이 질문에 얼마나 잘 답하는지를 1-5점으로 매기는 것입니다. 3점 미만일 경우 재생성이 필요합니다.',
 'relevance_grades': [],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

In [ ]:
# 컨텍스트를 만드는 전략 (단순히 join 등이 아니라)
  # 글자수 줄인다던가
  # k개 만큼 가져올 때 re-ranking 등

## 문서 관련성 평가 (grade_relevance)

In [ ]:
# retrieve -> grade_relevance -> generate

In [ ]:
# 실습 - grade_relevance 노드 만들기
  # retriever에서 document가 업데이트 되면 LLM에게 시켜서 document의 쿼리에 대한 연관성 평가, yes/no 답변하는 노드

In [ ]:
def grade_relevance_node(state: SelfRAGState) -> dict:

    relevance_prompt = ChatPromptTemplate.from_messages([
        ('system', '문서가 질문에 대한 답변에 도움이 된다면 "yes", 그렇지 않다면 "no"로 답변하세요.'),
        ('human', '질문: {question} \n\n문서 내용: {context}')
    ])

    relevance_results = []

    for doc in state['documents']:
        msg = relevance_prompt.format_messages(question=state['question'], context=doc.page_content)

        # 문자열(String)로 받기
        response_text = llm.invoke(msg).content.strip().lower()

        # 문자열 자체를 비교 (yes가 포함되어 있는지 확인)
        if 'yes' in response_text:
            relevance_results.append('yes')
        else:
            relevance_results.append('no')

    return {'relevance_grades': relevance_results}

In [ ]:
# 강사님 코드 수정
class _Relevance(BaseModel):
  binary : Literal['yes', 'no'] = Field(description = '문서가 질문과 관련이 있는지')

relevance_grader = llm.with_structured_output(_Relevance)

REL_PROMPT = ChatPromptTemplate.from_messages([
        ('system', '문서가 질문에 대한 답변에 도움이 된다면 "yes", 그렇지 않다면 "no"로만 답변하세요.'),
        ('human', '질문: {question} \n\n문서 내용: {context}')
    ])

def grade_relevance_node(state: SelfRAGState) -> dict:

    grades = []

    for d in state['documents']:
      # doc 대신 context로 인자 이름을 변경하여 REL_PROMPT의 {context}와 맞춥니다.
      msg = REL_PROMPT.format_messages(question = state['question'], context = d.page_content)
      grades.append(relevance_grader.invoke(msg).binary)

    return {'relevance_grades' : grades}

In [ ]:
builder = StateGraph(SelfRAGState)

builder.add_node('retrieve', retrieve_node)
builder.add_node('grade_relevance', grade_relevance_node)
builder.add_node('generate', generate_node)


builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_edge('grade_relevance', 'generate')
builder.add_edge('generate', END)
app = builder.compile()

In [ ]:
result = app.invoke({**INIT, 'question' : 'self-rag의 유용성 평가 기준은?'})

In [ ]:
result

{'question': 'self-rag의 유용성 평가 기준은?',
 'rewritten_question': '',
 'documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='2a6f31d6-9e76-447d-a6a3-6a07242cc066', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='32bfe861-8f3e-46f3-8a25-5a5aa6676cc7', metadata={'src': 'selfrag-usefulness'}, page_content='유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.')],
 'filtered_documents': [],
 'generation': 'self-RAG의 유용성 평가 기준은 답변이 질문에 정말 답하는지를 1-5점으로 평가하는 것입니다. 3점 미만이면 재생성합니다.',
 'relevance_grades': ['yes', 'yes', 'yes'],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

In [ ]:
result2 = app.invoke({**INIT, 'question' : '제로 콜라의 가격은?'})

In [ ]:
result2

{'question': '제로 콜라의 가격은?',
 'rewritten_question': '',
 'documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='2a6f31d6-9e76-447d-a6a3-6a07242cc066', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='e43cf4f6-58d4-4cb6-bb00-ba8ff31cdea8', metadata={'src': 'selfrag-relevance'}, page_content='관련성 평가에서 yes 문서만 컨텍스트로 추리고, 모두 no면 쿼리 재작성 또는 재검색.')],
 'filtered_documents': [],
 'generation': '모름',
 'relevance_grades': ['no', 'no', 'no'],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

## 조건부 재검색 (relevance_router)
- 관련 문서 있으면 generate, 없으면 다시 retrieve
- 현재는 무조건 generate함

In [ ]:
# retrieve -> grade_relevance -> router - 관련 문서 존재  -> generate
#   | <-------- 관련 문서 없음 <---- |

In [ ]:
class _Relevance(BaseModel):
  binary : Literal['yes', 'no'] = Field(description = '문서가 질문과 관련이 있는지')

relevance_grader = llm.with_structured_output(_Relevance)

def grade_and_filter_node(state: SelfRAGState) -> dict:
  grades = []
  keep = []

  for d in state['documents']:
    msg = REL_PROMPT.format_messages(question = state['question'], context = d.page_content)
    g = relevance_grader.invoke(msg).binary
    grades.append(g)

    if g == 'yes':
     keep.append(d)

  return {'relevance_grades': grades, 'filtered_documents' : keep}

def generate_filtered_node(state : SelfRAGState) -> dict:
  # 필터링된 문서들을 합쳐서 답변 생성
  ctx = '\n'.join(f" - {d.page_content}" for d in state['filtered_documents'])
  msg = RAG_PROMPT.format_messages(question = state['question'], context = ctx)

  return {'generation' : llm.invoke(msg).content}

def relevance_router(state : SelfRAGState) -> str:
  # 'filtered_document' -> 'filtered_documents'로 수정
  if state['filtered_documents']:
    return 'generate'
  else:
    return 'retrieve'

In [ ]:
builder = StateGraph(SelfRAGState)

builder.add_node('retrieve', retrieve_node)
builder.add_node('grade_relevance', grade_and_filter_node) # 필터링 노드로 수정
builder.add_node('generate', generate_filtered_node) # 필터링된 결과를 쓰는 생성 노드

builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_conditional_edges('grade_relevance', relevance_router, {'generate':'generate', 'retrieve':'retrieve'} )
builder.add_edge('generate', END)
app = builder.compile()

In [ ]:
result = app.invoke({**INIT, 'question' : 'self-rage의 관련성 평가는??'})

In [ ]:
result

{'question': 'self-rage의 관련성 평가는??',
 'rewritten_question': '',
 'documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='2a6f31d6-9e76-447d-a6a3-6a07242cc066', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='32bfe861-8f3e-46f3-8a25-5a5aa6676cc7', metadata={'src': 'selfrag-usefulness'}, page_content='유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.')],
 'filtered_documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='2a6f31d6-9e76-447d-a6a3-6a07242cc066', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.')],
 'generation': 'self-rage의 관련성 평가는 모름',
 'relevance_grades': ['yes', 'yes', 'no'],
 'hallucination_sco

### 재시도 카운터 (relevance_retries)

In [ ]:
def grade_and_filter_with_counter_node(state: SelfRAGState) -> dict:
  grades = []
  keep = []

  for d in state['documents']:
    msg = REL_PROMPT.format_messages(question = state['question'], context = d.page_content)
    g = relevance_grader.invoke(msg).binary
    grades.append(g)

    if g == 'yes':
     keep.append(d)

  return {'relevance_grades': grades, 'filtered_documents' : keep, 'relevance_retries': state['relevance_retries'] + 1}


def safe_relevance_router(state : SelfRAGState) -> str:
  # 'filtered_document' -> 'filtered_documents'로 수정
  if state['filtered_documents'] >= 2:
    return 'generate'

  return 'generate' if state['filtered_documents'] else 'retrieve'

In [ ]:
builder = StateGraph(SelfRAGState)

builder.add_node('retrieve', retrieve_node)
builder.add_node('grade_relevance', grade_and_filter_node) # 필터링 노드로 수정
builder.add_node('generate', generate_filtered_node) # 필터링된 결과를 쓰는 생성 노드

builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_conditional_edges('grade_relevance', relevance_router, {'generate':'generate', 'retrieve':'retrieve'} )
builder.add_edge('generate', END)
app = builder.compile()

## 쿼리 재작성 (rewrite_query_node)

In [ ]:
# retrieve -> grade_relevance -> router -> 연관 문서 있으면 -> generate -> END
#  ^ ---------- rewrite query ------------ <- 연관 문서 없으면

REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '다음 질문을 의미는 보존하되 검색이 더 잘되도록 동의어, 구체화로 재작성해줘. 한 줄로만'),
    ('human', '원래 질문: {question}')
])

def rewrite_query_node(state: SelfRAGState) -> dict:
  msg =REWRITE_PROMPT.format_messages(question = state['question'])
  new_q = llm.invoke(msg).content.strip()
  return {'rewritten_question' : new_q}

# 처음에는 state['question']에 들어있을 것이고, state['rewritten_question']은 빈 칸일것
  # 로직상 'rewritten_qustion'이 있다면 이걸 쓰도록 작성

def retrieve_aware_node(state: SelfRAGState) -> dict:
  q = state['rewritten_question'] or state['question']
  return {'documents' : retriever.invoke(q)}


def relevance_router(state : SelfRAGState) -> str:
  # 'filtered_document' -> 'filtered_documents'로 수정
  if state['filtered_documents']:
    return 'generate'
  return 'rewrite_query'

In [ ]:
builder = StateGraph(SelfRAGState)

builder.add_node('retrieve', retrieve_aware_node)
builder.add_node('grade_relevance', grade_and_filter_node)
builder.add_node('rewrite_query', rewrite_query_node)
builder.add_node('generate', generate_filtered_node)


builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_conditional_edges('grade_relevance', relevance_router, {'generate':'generate', 'rewrite_query':'rewrite_query'} )
builder.add_edge('rewrite_query', 'retrieve')
builder.add_edge('generate', END)
app = builder.compile()

In [ ]:
result = app.invoke({**INIT, 'question' : 'self-rag은?'})

In [ ]:
result

In [ ]:
result2 = app.invoke({**INIT, 'question' : 'zero콜라 가격은?'}) # 무한 루프 발생
# filtered_documents가 없으므로 query를 재작성 하는데
# 재작성 한 쿼리를 이용해도 관련 문서가 안나오는 현상 발생

KeyboardInterrupt: 

In [ ]:
result2

In [ ]:
# 쿼리 재작성 횟수 최대 1회로 제한


class SelfRAGState(TypedDict):
  question : str # 사용자 질문, state가 업데이트되더라도 question은 잘 안바뀜
  rewritten_question: str # 같은 쿼리를 같은 모델에다 던지면 비슷한 답변... 답변이 잘 안나온다면 query를 새로 작성해서 던짐(self-rag의 원래 논문에는 없음 (C-RAG, corrective rag))
  documents : List[Document] # 반환된 문서
  filtered_documents : List[Document] # 관련도 있는 문서 (필터됨)
  generation : str # 생성된 LLM 답변
  relevance_grades : List[str] # 문서별 관련 여부
  hallucination_score : str # 환각 여부
  usefulness_score : int # 1-5점, 유용성
  relevance_retries : int # 문서 검색 했으나 관련도 검사하여 필터링했으나, 개수가 부족하면 다시 검색
  generation_retries : int
  max_tries : int # 무한루프 방지


MAX_QUERY_REWRITE = 1

def rewrite_query_node(state: SelfRAGState) -> dict:
  msg =REWRITE_PROMPT.format_messages(question = state['question'])
  new_q = llm.invoke(msg).content.strip()
  return {
        "rewritten_question": new_q,
        "relevance_retries": state["relevance_retries"] + 1,
    }

# 처음에는 state['question']에 들어있을 것이고, state['rewritten_question']은 빈 칸일것
  # 로직상 'rewritten_qustion'이 있다면 이걸 쓰도록 작성

def retrieve_aware_node(state: SelfRAGState) -> dict:
  q = state['rewritten_question'] or state['question']
  return {'documents' : retriever.invoke(q)}


def relevance_router(state : SelfRAGState) -> str:
  # 'filtered_document' -> 'filtered_documents'로 수정
  if state["filtered_documents"]:
      return "generate"

  if state["relevance_retries"] < MAX_QUERY_REWRITE:
      return "rewrite_query"

  return "give_up"


def give_up_node(state: SelfRAGState) -> dict:
    return {
        "generation": "관련성이 충분한 문서를 찾지 못해 답변 생성을 중단했습니다."
    }

builder = StateGraph(SelfRAGState)

builder.add_node("retrieve", retrieve_aware_node)
builder.add_node("grade_relevance", grade_and_filter_node)
builder.add_node("rewrite_query", rewrite_query_node)
builder.add_node("generate", generate_filtered_node)
builder.add_node("give_up", give_up_node)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade_relevance")

builder.add_conditional_edges(
    "grade_relevance",
    relevance_router,
    {
        "generate": "generate",
        "rewrite_query": "rewrite_query",
        "give_up": "give_up",
    },
)

builder.add_edge("rewrite_query", "retrieve")
builder.add_edge("generate", END)
builder.add_edge("give_up", END)

app = builder.compile()


In [ ]:
result2 = app.invoke({**INIT, 'question' : 'zero콜라 가격은?'})

In [ ]:
result2

{'question': 'zero콜라 가격은?',
 'rewritten_question': '제로콜라의 가격은 얼마인가요?',
 'documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='e43cf4f6-58d4-4cb6-bb00-ba8ff31cdea8', metadata={'src': 'selfrag-relevance'}, page_content='관련성 평가에서 yes 문서만 컨텍스트로 추리고, 모두 no면 쿼리 재작성 또는 재검색.'),
  Document(id='75429129-00af-4d96-ad60-3d548f8699b3', metadata={'src': 'query-rewrite'}, page_content='쿼리 재작성은 검색 실패 시 동의어/구체화로 쿼리를 바꿔 다시 검색하는 폴백 패턴입니다.')],
 'filtered_documents': [],
 'generation': '관련성이 충분한 문서를 찾지 못해 답변 생성을 중단했습니다.',
 'relevance_grades': ['no', 'no', 'no'],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 1,
 'generation_retries': 0}

In [ ]:
# 강사님 코드
def rewrite_with_counter(state: SelfRAGState) -> dict:
  msg =REWRITE_PROMPT.format_messages(question = state['question'])
  return {"rewritten_question": llm.invoke(msg).content.strip(),"relevance_retries": state["relevance_retries"] + 1,}

def bounded_router(state : SelfRAGState) -> str:
  if state['filtered_documents']:
    return 'generate'
  if state['relevance_retries'] < 1:
    return 'rewrite_query'
  return 'generate'

## 환각 평가 (grade_hallucination)

In [ ]:
# retrieve -> grade_relevance -> router -> 연관 문서 있으면 -> generate -> grade_hallucination -> END
#  ^ ---------- rewrite query ------------ <- 연관 문서 없으면

# 우선 할루시네이션 판단 여부만 확인하는 노드 만들기

In [ ]:
class _Hallucination(BaseModel):
  binary : Literal['yes', 'no'] = Field(description = '답변이 환각인지 (yes = 문서 밖 정보, no = 근거 있음)')

hallucination_grader = llm.with_structured_output(_Hallucination)

HAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "당신은 엄격한 채점관입니다. 답변의 모든 사실이 문서에 명시적으로 기술되어 있어야 합니다. 답변이 주어진 문서에 근거하는지 평가해주세요. 문서 밖 정보면 yes(환각), 근거하면 no"),
    ("human", "문서: \n{context}\n\n답변: {answer}")

])

def grade_hallucination_node(state: SelfRAGState) -> dict:
  docs = state['filtered_documents'] or state['documents']
  ctx = '\n'.join(d.page_content for d in docs)
  msg = HAL_PROMPT.format_messages(context = ctx, answer = state['generation'])
  return {'hallucination_score': hallucination_grader.invoke(msg).binary}

In [ ]:
builder = StateGraph(SelfRAGState)

builder.add_node('retrieve', retrieve_aware_node)
builder.add_node('grade_relevance', grade_and_filter_node)
builder.add_node('rewrite_query', rewrite_query_node)
builder.add_node('generate', generate_filtered_node)
builder.add_node('grade_hallucination', grade_hallucination_node)


builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_conditional_edges('grade_relevance', relevance_router, {'generate':'generate', 'rewrite_query':'rewrite_query'} )
builder.add_edge('rewrite_query', 'retrieve')
builder.add_edge('generate', 'grade_hallucination')
builder.add_edge('grade_hallucination', END)
app = builder.compile()

In [ ]:
result3 = app.invoke({**INIT, 'question' : 'self-rag 환각 평가는 어떻게 하나요?'})

In [ ]:
result3

{'question': 'self-rag 환각 평가는 어떻게 하나요?',
 'rewritten_question': '',
 'documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='afc716b5-297f-43e6-b611-7777c39a00eb', metadata={'src': 'selfrag-hallucination'}, page_content='환각 평가는 답변이 문서에 근거하는지 확인합니다. yes(환각)면 다시 생성합니다.'),
  Document(id='2a6f31d6-9e76-447d-a6a3-6a07242cc066', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.')],
 'filtered_documents': [Document(id='8fffde83-c995-4c89-87ae-22ad63650b5c', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.')],
 'generation': '모름',
 'relevance_grades': ['yes', 'no', 'no'],
 'hallucination_score': 'no',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}